In [1]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage

In [2]:
env_path=Path.cwd().parent /".env"
load_dotenv(env_path)
api_key=os.getenv("GEMINI_API_KEY")
api_key2=os.getenv("GROQ_API_KEY")
api_key3=os.getenv("EXCHANGERATE_API_KEY")

if not api_key and  api_key2:
    print("API KEY MISSING")
    exit()
    
else:
    print("Initializing Model.....")

Initializing Model.....


Tool Creation

In [3]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated
@tool
def get_conversion_factor(base_currency :str,target_currency :str) -> float:
    """This function fetches the curency conversion factor 
    btn base currency and targted currency
    """
    
    url=f"https://api.fastforex.io/fetch-one?from={base_currency}&to={target_currency}&api_key={api_key3}"
    try:
        response=requests.get(url)
        response.raise_for_status()
        return response.json()
    
    except Exception as e:
        print(f'Error is {e}')
        
@tool
def convert(base_currency_value:float , conversion_rate:float) -> float:
    """This function multiples the given money(currency amount) as per the exchange or conversion rate"""
    return base_currency_value*conversion_rate

In [4]:
result=get_conversion_factor.invoke({'base_currency':'USD','target_currency':'NPR'})


In [5]:
result

{'base': 'USD',
 'result': {'NPR': 153.356},
 'updated': '2026-09-24T06:12:08Z',
 'ms': 8}

In [6]:
base_currency_info=result['result']
conversion_factor=list(base_currency_info.values())[0]

In [7]:
convert.invoke({'base_currency_value':25.00,'conversion_rate':conversion_factor})

3833.8999999999996

Binding With Llm

In [8]:
llm=ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite',api_key=api_key)

In [9]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert])


In [10]:
query="What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?"

In [11]:
messages=[HumanMessage(query)]

In [12]:
messages

[HumanMessage(content='What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?', additional_kwargs={}, response_metadata={})]

In [13]:
ai_message=llm_with_tools.invoke(messages)



In [14]:
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'call_122220': 'EnEKbwFpFH0TdUrF62cQqjm7HZnmKATmSt8CgF0BBQPMoNf/88m3SXxVZR9CYNNhVkVk2tA65/pxoqT4NdmwBw2RgbaUZhuj5rhu1xaWkvpBEQTbARW8XbYmuUMRgu6MjiSZlUTtEVdEOfHx/dZMvSNiDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d212-12f8-7190-8a64-7ca261d304d2-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_122220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 29, 'total_tokens': 211, 'input_token_details': {'cache_read': 0}})

In [15]:
messages.append(ai_message)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_122220',
  'type': 'tool_call'}]

In [16]:
ai_message.tool_calls[0]['args']

{'base_currency': 'USD', 'target_currency': 'NPR'}

In [17]:
tool_msg_1=get_conversion_factor.invoke(ai_message.tool_calls[0]['args'])
tool_msg_1

{'base': 'USD',
 'result': {'NPR': 153.356},
 'updated': '2026-09-24T06:12:08Z',
 'ms': 3}

In [18]:
import json
# for tool_call in ai_message.tool_calls:
    #execute the 1st tool and get the value of conversion rate
    #execute the 2nd tool using the conversion rate from tool 1
# if tool_call['name']=='get_conversion_factor':
tool_msg_1=get_conversion_factor.invoke(ai_message.tool_calls[0]['args'])
        #fetch this convesion rate 
base_currency_info=tool_msg_1['result']
conversion_rate=list(base_currency_info.values())[0]
        #append this conversion rate from too1 to messages
messages.append(ToolMessage(content=str(tool_msg_1),tool_call_id=ai_message.tool_calls[0]['id']))
        
    #execute the 2nd tool using the conversion rate from tool 1
    
    
#if tool_call['name']=='convert':
        #fetch


In [19]:

# tool_msg_2=convert.invoke(tool_call)
# messages.append(tool_msg_2)

In [20]:
messages

[HumanMessage(content='What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'call_122220': 'EnEKbwFpFH0TdUrF62cQqjm7HZnmKATmSt8CgF0BBQPMoNf/88m3SXxVZR9CYNNhVkVk2tA65/pxoqT4NdmwBw2RgbaUZhuj5rhu1xaWkvpBEQTbARW8XbYmuUMRgu6MjiSZlUTtEVdEOfHx/dZMvSNiDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d212-12f8-7190-8a64-7ca261d304d2-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_122220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 29, 'total_tokens': 211, '

In [21]:
messages.append(HumanMessage(query))


In [22]:
messages

[HumanMessage(content='What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'call_122220': 'EnEKbwFpFH0TdUrF62cQqjm7HZnmKATmSt8CgF0BBQPMoNf/88m3SXxVZR9CYNNhVkVk2tA65/pxoqT4NdmwBw2RgbaUZhuj5rhu1xaWkvpBEQTbARW8XbYmuUMRgu6MjiSZlUTtEVdEOfHx/dZMvSNiDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d212-12f8-7190-8a64-7ca261d304d2-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_122220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 29, 'total_tokens': 211, '

In [23]:
ai_message_2=llm_with_tools.invoke(messages)

In [24]:
messages.append(ai_message_2)
ai_message_2.tool_calls

[{'name': 'convert',
  'args': {'conversion_rate': 153.356, 'base_currency_value': 10},
  'id': 'call_250017',
  'type': 'tool_call'}]

In [25]:
messages

[HumanMessage(content='What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'call_122220': 'EnEKbwFpFH0TdUrF62cQqjm7HZnmKATmSt8CgF0BBQPMoNf/88m3SXxVZR9CYNNhVkVk2tA65/pxoqT4NdmwBw2RgbaUZhuj5rhu1xaWkvpBEQTbARW8XbYmuUMRgu6MjiSZlUTtEVdEOfHx/dZMvSNiDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d212-12f8-7190-8a64-7ca261d304d2-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_122220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 29, 'total_tokens': 211, '

In [26]:
tool_msg_2=convert.invoke(ai_message_2.tool_calls[0]['args'])

tool_msg_2

1533.56

In [27]:
messages.append(ToolMessage(content=tool_msg_2,tool_call_id=ai_message_2.tool_calls[0]['id']))

In [28]:
messages


[HumanMessage(content='What is the conversion factor between usd and npr, and on that basis, how much is $10 USD in npr?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "NPR"}'}, '__gemini_function_call_thought_signatures__': {'call_122220': 'EnEKbwFpFH0TdUrF62cQqjm7HZnmKATmSt8CgF0BBQPMoNf/88m3SXxVZR9CYNNhVkVk2tA65/pxoqT4NdmwBw2RgbaUZhuj5rhu1xaWkvpBEQTbARW8XbYmuUMRgu6MjiSZlUTtEVdEOfHx/dZMvSNiDw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d212-12f8-7190-8a64-7ca261d304d2-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_122220', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 182, 'output_tokens': 29, 'total_tokens': 211, '

In [29]:
messages.append(HumanMessage(query ))
ans=llm_with_tools.invoke(messages)

In [30]:
ans.content[0]['text']

'The conversion factor from USD to NPR is 153.356. Based on this rate, $10 USD is equivalent to 1,533.56 NPR.'